# Leaf Wood Segmentation

build on top of https://github.com/qforestlab/leaf-wood-segmentation-with-deep-learning to perform the segmentation

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
#| default_exp lw_segmentation

## Download weights

In [ ]:
#| export
from pyprojroot import here
import requests
from pathlib import Path
import zipfile
import laspy
import numpy as np

In [ ]:
#| export
url = "https://zenodo.org/records/13767795/files/model_weights.zip?download=1"
weights_dir = here("data/weights")
dst = weights_dir / "model_weights.zip"

In [ ]:
#| export
if not dst.exists():
    with requests.get(url, stream=True) as r:
        r.raise_for_status()
        with open(dst, "wb") as f:
            for chunk in r.iter_content(chunk_size=8192):
                if chunk:
                    f.write(chunk)

if not weights_dir.exists():
    weights_dir.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(dst, "r") as z:
        z.extractall(weights_dir)

In [ ]:
#| export
def read_cloud(path):
    las = laspy.read(path)
    return {
        'point': las.xyz,
        'point_orig': np.stack((las.X, las.Y, las.Z), axis=-1), # UNSCALEd points to avoid float precision issues
        'header': las.header
    }

In [ ]:
tile0 = read_cloud(here("data/1_tiled/tile_1.laz"))

In [ ]:
tile0['point'] = tile0['point'][:1000] #subset for testing
tile0['point_orig'] = tile0['point_orig'][:1000] #subset for testing

In [ ]:
tile0['point'].shape

(1000, 3)

## Inference Point Tranformer

custom point transformer for our inference usecase

In [ ]:
#| export
import open3d.ml.torch as ml3d
import torch
from sklearn.neighbors import KDTree

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


In [ ]:
#| export
class LWTransformerBatch:
    def __init__(self, batches):
        pc = []
        point_inds = []
        splits = [0]

        for batch in batches:
            data = batch['data']
            pc.append(data['point'])
            point_inds.append(data['point_inds'])
            splits.append(splits[-1] + data['point'].shape[0])

        self.point = torch.cat(pc, 0)  
        self.point_inds = torch.cat(point_inds, 0)
        self.row_splits = torch.LongTensor(splits)
        self.feat = [] # expected by the model forward

    def pin_memory(self):
        self.point = self.point.pin_memory()
        return self
    
    def to(self, device):
        self.point = self.point.to(device)

class LWBatcher(ml3d.dataloaders.ConcatBatcher):
    def collate_fn(self, batches):
        return {'data': LWTransformerBatch(batches), 'attr': {}}

In [ ]:
#| export
class LWPointTransformer(ml3d.models.PointTransformer):
    def transform(self, data, attr):
        pc = data['point'].copy()
        label = data['label'].copy()
        tree = data['search_tree']

        # Select num_points from whole cloud (randomly pick center and take num_point nearest neighbours) 
        pc, selected_idxs, center_point = self.trans_point_sampler(
            pc=pc,
            feat=None,
            label=label,
            search_tree=tree,
            num_points=self.cfg.max_voxels
        )
        label = label[selected_idxs]
        # Apply augmentations
        augment_cfg = self.cfg.get('augment', {}).copy()
        val_augment_cfg = {}
        if 'recenter' in augment_cfg:
            val_augment_cfg['recenter'] = augment_cfg.pop('recenter')
        if 'normalize' in augment_cfg:
            val_augment_cfg['normalize'] = augment_cfg.pop('normalize')

        pc, _, label = self.augmenter.augment(pc, None, None, val_augment_cfg)

        inputs = dict()
        inputs['point'] = torch.from_numpy(pc).to(torch.float32)
        inputs['point_inds'] = torch.from_numpy(selected_idxs).to(torch.int64)

        return inputs

In [ ]:
infer_dataset = ml3d.datasets.InferenceDummySplit(tile0)
infer_sampler = infer_dataset.sampler

In [ ]:
#| export
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

class LWSemanticSegmentation(ml3d.pipelines.SemanticSegmentation):
    def run_inference(self, data):
        """Run inference on given data.

        Args:
            data: A raw data.

        Returns:
            Returns the inference results.
        """
        cfg = self.cfg
        model = self.model
        device = self.device

        model.to(device)
        model.device = device
        model.eval()

        batcher = LWBatcher(self.device, "LWPointTransformer")
        infer_dataset = ml3d.datasets.InferenceDummySplit(data)
        self.dataset_split = infer_dataset
        infer_sampler = infer_dataset.sampler
        infer_split = ml3d.dataloaders.TorchDataloader(dataset=infer_dataset,
                                      preprocess=model.preprocess,
                                      transform=model.transform,
                                      sampler=infer_sampler,
                                      use_cache=False)
        infer_loader = DataLoader(infer_split,
                                  batch_size=cfg.batch_size,
                                  sampler=ml3d.dataloaders.get_sampler(infer_sampler),
                                  collate_fn=batcher.collate_fn)

        model.trans_point_sampler = infer_sampler.get_point_sampler()
        # self.curr_cloud_id = -1
        # self.test_probs = []
        # self.ori_test_probs = []
        # self.ori_test_labels = []
        with torch.no_grad():
            inputs = next(iter(infer_loader))
            inputs['data'].to(device)
            return model(inputs['data'])
        


    #         for inputs in infer_loader:
    #             inputs['data'].to(device)
    #             results = model(inputs['data'])
    #             self.update_tests(infer_sampler, inputs, results)

    #     inference_result = {
    #         'predict_labels': self.ori_test_labels.pop(),
    #         'predict_scores': self.ori_test_probs.pop()
    #     }

    #     return inference_result

    
    # def update_tests(self, sampler, inputs, results):
    #     """Update tests using sampler, inputs, and results."""
    #     split = sampler.split # split = 'test'
    #     end_threshold = 0.5
    #     if self.curr_cloud_id != sampler.cloud_id:
    #         self.curr_cloud_id = sampler.cloud_id
    #         num_points = sampler.possibilities[sampler.cloud_id].shape[0]
    #         self.pbar = tqdm(
    #             total=num_points,
    #             desc="{} {}/{}".format(split, self.curr_cloud_id, len(sampler.dataset))
    #         )
    #         self.pbar_update = 0
    #         self.test_probs.append(np.zeros(shape=[num_points, self.model.cfg.num_classes], dtype=np.float16))
    #         self.complete_infer = False

    #     this_possiblility = sampler.possibilities[sampler.cloud_id]
    #     self.pbar.update(this_possiblility[this_possiblility > end_threshold].shape[0] - self.pbar_update)
    #     self.pbar_update = this_possiblility[this_possiblility > end_threshold].shape[0]

    #     self.test_probs[self.curr_cloud_id] = self.model.update_probs(
    #         inputs,
    #         results,
        #     self.test_probs[self.curr_cloud_id],
        # )

        # # When all predictions are made
        # if (split in ['test'] and
        #         this_possiblility[this_possiblility > end_threshold].shape[0]
        #         == this_possiblility.shape[0]):

        #     proj_inds = self.model.preprocess(
        #         self.dataset_split.get_data(self.curr_cloud_id),
        #         {'split': split},
        #     ).get('proj_inds', None)
            
        #     if proj_inds is None:
        #         proj_inds = np.arange(self.test_probs[self.curr_cloud_id].shape[0])

        #     test_labels = np.argmax(self.test_probs[self.curr_cloud_id][proj_inds], 1)

        #     self.ori_test_probs.append(self.test_probs[self.curr_cloud_id][proj_inds])
        #     self.ori_test_labels.append(test_labels)
        #     self.complete_infer = True

number of 2 cms voxels in a cube that is 10x10x30 meters

In [ ]:
n_voxel_tile = 10 / 0.02  * 10 / 0.02  * 30 / 0.02
n_voxel_tile

375000000.0

In [ ]:
#| export
model_config = {'name': 'PointTransformer',
 'batcher': 'ConcatBatcher',
 'ckpt_path': str(here("data/weights/model_weights/weights_pointtransformer.pth")),
 'is_resume': False,
 'in_channels': 3,
 'blocks': [2, 3, 4, 6, 3],
 'num_classes': 2,
 'voxel_size': 0.02,
#  'max_voxels': 2_000_000,
 'max_voxels': 10_000, # for testing
 'ignored_label_inds': [],
 'augment': {
    'recenter' : {'dim': [0, 1, 2]},
    'normalize' : {},
    'rotate':
      {'method': 'vertical'},
    'scale': {
      'min_s': 0.9,
      'max_s': 1.1
    }
 }
}

In [ ]:
#| export
pipeline_config = {
'name': 'SemanticSegmentation',
 'device': 'cuda:0',
 'num_workers': 16,
 'batch_size': 1,
}

In [ ]:
#| export
model = LWPointTransformer(**model_config)
pipeline = LWSemanticSegmentation(model, **pipeline_config)

In [ ]:
#| export
pipeline.load_ckpt(here("data/weights/model_weights/weights_pointtransformer.pth"))

In [ ]:
predictions = pipeline.run_inference(tile0)

## Post processing

- split wood/leaves 
- I don't fully understand what's going on, but since is sampling data (I think the max_voxels params really matters) I need to return it back to the original point using a kdtree


In [ ]:
proj_inds = model.preprocess(tile0, attr={'split': 'test'}).get('proj_inds', None)
proj_inds.shape

(1000,)

In [ ]:
probs = torch.nn.functional.softmax(predictions, dim=-1).cpu().data.numpy()

In [ ]:
probs_points = probs[proj_inds]

In [ ]:
probs_points.shape

(1000, 2)

In [ ]:
is_wood = probs_points[:, 0] > 0.75
is_wood, is_wood.shape

(array([False,  True, False, False, False,  True, False, False, False,
        False,  True,  True,  True, False, False,  True, False, False,
        False, False, False, False,  True,  True, False, False, False,
        False, False, False, False, False, False, False, False, False,
        False,  True,  True, False,  True, False, False, False, False,
        False, False, False, False,  True, False, False, False, False,
        False, False, False, False, False, False,  True,  True, False,
        False,  True,  True,  True,  True,  True, False,  True, False,
        False, False, False, False, False, False, False,  True,  True,
         True, False, False, False, False, False, False, False, False,
         True,  True, False, False, False, False,  True,  True,  True,
        False, False, False,  True, False, False,  True,  True, False,
        False, False, False, False, False,  True, False, False, False,
        False, False, False, False,  True, False, False, False, False,
      

In [ ]:
cloud_leaves = tile0['point'][~is_wood]
cloud_leaves.shape

(748, 3)

In [ ]:
cloud_wood = tile0['point'][is_wood]
cloud_wood.shape

(252, 3)

In [ ]:
#| export
def post_processing(pred, cloud, threshold) -> tuple[np.ndarray, np.ndarray]:
    probs = torch.nn.functional.softmax(pred, dim=-1).cpu().data.numpy()
    # conver to original points
    proj_inds = model.preprocess(cloud, attr={'split': 'test'})['proj_inds']
    probs_points = probs[proj_inds]
    # softmax and threshold
    is_wood = probs_points[:, 0] > threshold
    # return the unscaled dimensions not the input
    return (cloud['point_orig'][is_wood], cloud['point_orig'][~is_wood])

In [ ]:
post_processing(predictions, tile0, 0.75)

(array([[-244893, -162273, -261211],
        [-244593, -162287, -261320],
        [-244720, -162311, -261162],
        [-244635, -162368, -261243],
        [-244640, -162268, -261114],
        [-244558, -162279, -260985],
        [-243698, -162383, -260662],
        [-243644, -162405, -260668],
        [-235200, -162895, -263514],
        [-235107, -162782, -263533],
        [-234760, -162757, -263448],
        [-234296, -162981, -263221],
        [-235800, -162705, -262250],
        [-235623, -162928, -262055],
        [-235657, -162546, -262141],
        [-235640, -162447, -261948],
        [-235679, -162415, -261939],
        [-234856, -162446, -262807],
        [-234677, -162173, -262721],
        [-233946, -163028, -263491],
        [-235682, -162343, -261877],
        [-235626, -162362, -261860],
        [-235630, -162288, -261805],
        [-235400, -162422, -261810],
        [-235409, -162364, -261792],
        [-235580, -162343, -261456],
        [-235174, -162178, -261550],
 

## Save point clouds

In [ ]:
#| export
def save_cloud(cloud: np.ndarray, name, header):
    las = laspy.LasData(header)
    las.X = cloud[:, 0]
    las.Y = cloud[:, 1]
    las.Z = cloud[:, 2]
    las.write(name)

## Complete processing

In [ ]:
#| export
import fastcore.xtras # ls
from tqdm.auto import tqdm

In [ ]:
#| export
cloud_paths = here("data/1_tiled/").ls()
cloud_paths

(#1512) [Path('/Stor3/simone/semantic-segmentation/data/1_tiled/tile_1069.laz'),Path('/Stor3/simone/semantic-segmentation/data/1_tiled/tile_124.laz'),Path('/Stor3/simone/semantic-segmentation/data/1_tiled/tile_16.laz'),Path('/Stor3/simone/semantic-segmentation/data/1_tiled/tile_1107.laz'),Path('/Stor3/simone/semantic-segmentation/data/1_tiled/tile_607.laz'),Path('/Stor3/simone/semantic-segmentation/data/1_tiled/tile_216.laz'),Path('/Stor3/simone/semantic-segmentation/data/1_tiled/tile_137.laz'),Path('/Stor3/simone/semantic-segmentation/data/1_tiled/tile_307.laz'),Path('/Stor3/simone/semantic-segmentation/data/1_tiled/tile_441.laz'),Path('/Stor3/simone/semantic-segmentation/data/1_tiled/tile_12.laz'),Path('/Stor3/simone/semantic-segmentation/data/1_tiled/tile_1257.laz'),Path('/Stor3/simone/semantic-segmentation/data/1_tiled/tile_1268.laz'),Path('/Stor3/simone/semantic-segmentation/data/1_tiled/tile_847.laz'),Path('/Stor3/simone/semantic-segmentation/data/1_tiled/tile_354.laz'),Path('/St

In [ ]:
model.cfg.max_voxels = 100_000 # production value, the time grows exponentially with voxel size ...

In [ ]:
#| export
dir_leaves = here("data/2_lw_segmented/leaves/")
dir_wood = here("data/2_lw_segmented/wood/")
dir_wood.mkdir(parents=True, exist_ok=True)
dir_leaves.mkdir(parents=True, exist_ok=True)

In [ ]:
#| export
def process_cloud(path):
    cloud = read_cloud(path)
    predictions = pipeline.run_inference(cloud)
    wood, leaves = post_processing(predictions, cloud, 0.75)
    save_cloud(leaves, dir_leaves / Path(path).name, cloud['header'])
    save_cloud(wood, dir_wood / Path(path).name, cloud['header'])

In [ ]:
process_cloud(cloud_paths[0])

ValueError: could not broadcast input array from shape (191072,) into shape (445824,)

In [ ]:
cloud = read_cloud(cloud_paths[0])
pred = pipeline.run_inference(cloud)

In [ ]:
probs = torch.nn.functional.softmax(pred, dim=-1).cpu().data.numpy()


In [ ]:
probs.shape

(100000, 2)

In [ ]:
proj_inds = model.preprocess(cloud, attr={'split': 'test'})['proj_inds']

In [ ]:
proj_inds.shape

(445824,)

In [ ]:
proj_inds.max()

47646

In [ ]:

proj_inds = np.squeeze(
    search_tree.query(points, return_distance=False))
proj_inds = proj_inds.astype(np.int32)
# Clamp indices to valid range
proj_inds = np.clip(proj_inds, 0, probs.shape[0] - 1)
probs_points = probs[proj_inds]
# softmax and threshold
is_wood = probs_points[:, 0] > threshold
# return the unscaled dimensions not the input
# return (cloud['point_orig'][is_wood], cloud['point_orig'][~is_wood])

In [ ]:
#| export
def process_all(paths):
    for path in tqdm(paths):
        process_cloud(path)